In [ ]:
# SQL block → df_6
df_6 = _datapad_sql("""
USE careloopclinic;
""")

In [ ]:
# SQL block → df_7
df_7 = _datapad_sql("""
SELECT * FROM appointments LIMIT 5;
""")

In [ ]:
# SQL block → df_8
df_8 = _datapad_sql("""
SELECT c.clinic_name, c.city, COUNT(a.appointment_id) as Total_Appointments, 
ROUND(SUM(a.no_show)/ COUNT(a.appointment_id)*100,1) as No_Show_Pct
FROM appointments as a
LEFT JOIN clinics as c
ON a.clinic_id = c.clinic_id
GROUP BY c.clinic_name, c.city
-- HAVING COUNT(a.appointment_id) > 95
ORDER BY No_Show_Pct DESC;


""")

In [ ]:
import pandas as pd

In [ ]:
df_appt = pd.read_csv(r"C:\Users\CoreX\OneDrive\Desktop\CareLoop Clinics\Data\appointments_clean.csv")
df_clinic = pd.read_csv(r"C:\Users\CoreX\OneDrive\Desktop\CareLoop Clinics\Data\clinics_temp.csv")

In [ ]:
df_combine = pd.merge(left= df_appt, 
         right= df_clinic,
        on= 'clinic_id',
        how= 'left')

In [ ]:
def no_show_pct(series):
    return (series.sum() / series.count() * 100)
df_combine.groupby(by=['clinic_name', 'city']).agg(
    Total_Appointments = ('appointment_id', 'count'), 
    No_Show_PCT = ('no_show', no_show_pct)).sort_values('No_Show_PCT', ascending=False)

In [ ]:
# SQL block → df_9
df_9 = _datapad_sql("""

""")

In [ ]:
# SQL block → df_10
df_10 = _datapad_sql("""
WITH app_pvt AS (SELECT a.no_show, a.lead_days,
CASE 
WHEN a.lead_days <= 7 THEN '1-7 days'
WHEN a.lead_days <= 14 THEN '8-14 days'
WHEN a.lead_days <= 30 THEN '15-30 days'
ELSE '31+ days'
END as lead_day_buckets
FROM appointments as a
ORDER BY lead_days)

SELECT lead_day_buckets, 
COUNT(no_show) as Total_Appointments,
SUM(no_show) as Show_Appointments,
(100 - SUM(no_show) / COUNT(no_show) * 100) as No_Show_Pct
FROM app_pvt
GROUP BY lead_day_buckets
ORDER BY MIN(lead_days)

""")

In [ ]:
# SQL block → df_11
df_11 = _datapad_sql("""
SELECT COUNT(a.no_show) as Total_Appointments, 
SUM(a.no_show) as Show_Appointments, 
(100 - SUM(a.no_show)/ COUNT(a.no_show) * 100) as No_Show_Pct,
CASE 
WHEN a.lead_days <= 7 THEN '1-7 days'
WHEN a.lead_days <= 14 THEN '8-14 days'
WHEN a.lead_days <= 30 THEN '15-30 days'
ELSE '31+ days'
END as lead_day_buckets
FROM appointments as a
GROUP BY lead_day_buckets 
ORDER BY MIN(a.lead_days)
""")

In [ ]:
# SQL block → df_12
df_12 = _datapad_sql("""
WITH monthly as 
(
    SELECT SUBSTR(appt_date,1,7) as month, 
    COUNT(no_show) as Total_Appointments, 
    (100 - SUM(no_show)/ COUNT(no_show) * 100) as No_Show_Pct
    FROM appointments
    GROUP BY month
) 

SELECT *, 
    LAG(No_Show_Pct, 1, "-") OVER (ORDER BY month) as Pre_No_Show_Pct,
    (No_Show_Pct - LAG(No_Show_Pct, 1, "-") OVER (ORDER BY month)) as Pct_Change 
FROM monthly
ORDER BY month
""")

In [ ]:
# SQL block → df_13
df_13 = _datapad_sql("""
SELECT 
c.city, 
c.clinic_name,
ROUND(SUM(a.no_show)/ COUNT(a.appointment_id)*100.0,1) as No_Show_Pct,
RANK() OVER (
    PARTITION BY c.city 
    ORDER BY ROUND(SUM(a.no_show) * 100.0 / COUNT(a.appointment_id),1) DESC
) as City_Rank
FROM appointments as a
LEFT JOIN clinics as c
ON a.clinic_id = c.clinic_id
GROUP BY c.clinic_name, c.city
ORDER BY c.city, City_Rank ASC;
""")

In [ ]:
# SQL block → df_14
df_14 = _datapad_sql("""

""")